In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '16'
# 模块导入
# from Train.train_utils import (
#     set_seed
# )

In [2]:
# # 设置随机种子
# set_seed(42)

In [3]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

使用设备: cuda


#### LOB TRADE数据预处理
- 1.补全时间戳，补充成固定100ms间隔的数据
- 2.生成label

In [4]:
from pathlib import Path
save_dir = Path('/root/autodl-tmp/train_data/')
save_dir.mkdir(parents=True,exist_ok=True)

In [5]:

# #### 特征工程
# from Data_Pipeline.preprocessors.lob_data_process import process_lob_data
# from Data_Pipeline.preprocessors.trade_data_process import process_trade_data
# lob_data_dir = '/root/autodl-tmp/ETHUSDT/20levels_parquet'
# trade_data_dir = '/root/autodl-tmp/ETHUSDT/trade'


# # date = ['2025-11-04','2025-11-05','2025-11-06','2025-11-07','2025-11-08','2025-11-09','2025-11-10',
# #         '2025-11-11','2025-11-12','2025-11-13','2025-11-14','2025-11-15','2025-11-16','2025-11-17',
# #         '2025-11-18','2025-11-19','2025-11-20','2025-11-21','2025-11-22','2025-11-23','2025-11-24',
# #         '2025-11-25','2025-11-26','2025-11-27','2025-11-28','2025-11-29','2025-11-30','2025-12-01',
# #         '2025-12-02','2025-12-03','2025-12-04','2025-12-05','2025-12-06','2025-12-07',
# #         ]
# date  = ['2025-12-08','2025-12-09','2025-12-10','2025-12-11','2025-12-12','2025-12-13',
#          '2025-12-14','2025-12-15','2025-12-16','2025-12-17','2025-12-18','2025-12-19',
#          '2025-12-20','2025-12-21','2025-12-22','2025-12-23']
# levels = 10


# lob_data = process_lob_data(data_dir=lob_data_dir,date = date,levels=levels)
# trade_data = process_trade_data(data_dir=trade_data_dir,date = date,window_ms=100)

# lob_data.write_parquet(save_dir / 'LOB_ETHUSDT_test.parquet')
# trade_data.write_parquet(save_dir / 'Trade_ETHUSDT_test_agg100ms.parquet')


In [6]:
# lob_data_path = save_dir / 'LOB_ETHUSDT_test.parquet'
# trade_data_path = save_dir / 'Trade_ETHUSDT_test_agg100ms.parquet'
# trade_data = pl.read_parquet(trade_data_path)
# lob_data = pl.read_parquet(lob_data_path)
# from Data_Pipeline.preprocessors.trade_data_process import align_trade_with_lob
# trade_data = align_trade_with_lob(trade_data = trade_data,full_lob_data = lob_data)

In [ ]:

# import numpy as np
# from Data_Pipeline.generators.label_gen import generate_data_dict
# lob_data_path = save_dir / 'LOB_ETHUSDT_test.parquet'
# trade_data_path = save_dir / 'Trade_ETHUSDT_test_agg100ms.parquet'

# label_window = 300
# levels = 10
# change_window = None # none 就是对即时的价格进行预测
# need_price = True

# data_dict,trade_labels_ret,price_data,time_bucket = generate_data_dict(lob_data_path,
#                                                                         trade_data_path,
#                                                                         levels=levels,
#                                                                         label_window=label_window,
#                                                                         need_price=need_price)

# ## 存储为npy格式
# np.save(save_dir / 'lob_data.npy',data_dict['lob'])
# np.save(save_dir / 'trade_data.npy',data_dict['trade'])
# np.save(save_dir / 'trade_labels_ret.npy',trade_labels_ret)
# if price_data is not None:
#     np.save(save_dir / 'price.npy',price_data)
# if time_bucket is not None:
#     np.save(save_dir / 'time_bucket.npy',time_bucket)


In [9]:
# np.isnan(trade_labels_ret).sum()

## 加载处理好的本地数据

In [10]:

import numpy as np
## 读取

lob_data = np.load('/root/autodl-tmp/train_data/lob_data.npy')
trade_data = np.load('/root/autodl-tmp/train_data/trade_data.npy')
labels_ret = np.load('/root/autodl-tmp/train_data/trade_labels_ret.npy')

print(f"trade_data.shape: {trade_data.shape}")
print(f"lob_data.shape: {lob_data.shape}")
print(f"labels_ret.shape: {labels_ret.shape}")

alpha= 0.0003
labels_class = np.ones_like(labels_ret, dtype=np.int8)
# 3. 向量化赋值：涨→2，跌→0
# 涨：labels_ret > alpha
labels_class[labels_ret > alpha] = 2
# 跌：labels_ret < -alpha
labels_class[labels_ret < -alpha] = 0

trade_data.shape: (29375696, 23)
lob_data.shape: (29375696, 40)
labels_ret.shape: (29375696,)


In [11]:
# def rearrange_lob_data(lob_data):
#     """
#     输入：lob_data = (N, 4, 10)
#           [askprice, bidprice, askvol, bidvol]
#           每一个是 [1~10档]

#     输出：(N, 2, 20)
#           通道0：价格 [bid10, bid9,...bid1, ask1,...ask9,ask10]
#           通道1：量   [bid10, bid9,...bid1, ask1,...ask9,ask10]
#     """
#     N = lob_data.shape[0]

#     # 取出原始通道
#     ask1_10_price = lob_data[:, 0, :]  # (N,10)
#     bid1_10_price = lob_data[:, 1, :]
#     ask1_10_vol   = lob_data[:, 2, :]
#     bid1_10_vol   = lob_data[:, 3, :]

#     # ==================== 价格重排 ====================
#     # BID 反转：bid10 ~ bid1
#     bid10_1_price = np.flip(bid1_10_price, axis=-1)  # (N,10)
#     # ASK 保持正序：ask1 ~ ask10
#     ask1_10_price = ask1_10_price

#     # 拼接最终价格序列
#     price_final = np.concatenate([bid10_1_price, ask1_10_price], axis=-1)  # (N,20)

#     # ==================== 量重排 ====================
#     bid10_1_vol = np.flip(bid1_10_vol, axis=-1)
#     ask1_10_vol = ask1_10_vol
#     vol_final = np.concatenate([bid10_1_vol, ask1_10_vol], axis=-1)

#     # ==================== 组合成 (N,2,20) ====================
#     output = np.stack([price_final, vol_final], axis=1)  # (N,2,20)

#     return output

#     ## 把 lob_data 的ask_price和bid_price的合并为1个channel
# lob_data = rearrange_lob_data(lob_data)
# print(f"after rearrange, lob_data.shape: {lob_data.shape}")


In [12]:
np.unique(labels_class,return_counts=True)[1] / np.unique(labels_class,return_counts=True)[1].sum()

array([0.19840762, 0.60682586, 0.19476652])

In [13]:
import polars as pl
pl.Series(labels_ret).abs().describe()

statistic,value
str,f64
"""count""",2.9375696e7
"""null_count""",0.0
"""mean""",0.00035
"""std""",0.000415
"""min""",1.8421e-12
"""25%""",0.000096
"""50%""",0.000224
"""75%""",0.00045
"""max""",0.012228


### 加载和保存config

In [14]:
from Utils.config_loader import load_config
from pathlib import Path
print("加载配置...")
model_config_path = '/root/lio/Trade_LOB_MultiModal/Configs/model_config.yaml'
train_config_path = '/root/lio/Trade_LOB_MultiModal/Configs/train_config.yaml'
model_config = load_config(model_config_path)
train_config = load_config(train_config_path)

import yaml
model_version = model_config.get('model_version', 'multi_modal_model_1')
description = model_config.get('description')
print(f"model_version: {model_version}")
print(f"description: {description}")



config_save_path = f"/root/lio/Trade_LOB_MultiModal/checkpoints/{model_version}"
config_save_path = Path(config_save_path)
# 确保保存目录存在
config_save_path.mkdir(exist_ok=True,parents=True)
## 保存model的config
model_config_save_path = f"{config_save_path}/model_config.yaml"
train_config_save_path = f"{config_save_path}/train_config.yaml"

with open(model_config_save_path, 'w') as f:
    yaml.dump(model_config, f, indent=4, sort_keys=False, allow_unicode=True)
with open(train_config_save_path, 'w') as f:
    yaml.dump(train_config, f, indent=4, sort_keys=False, allow_unicode=True)

加载配置...
model_version: multi_modal_model_3
description: 1.deeplob编码 2.使用的是trade的即时标签进行训练 3.lob数据是1*40


### 创建数据集

In [15]:
import yaml
all_config = '/root/lio/Trade_LOB_MultiModal/Configs/experiment_config.yaml'
with open(all_config, 'r') as f:
    all_config = yaml.safe_load(f)
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
## 创建数据集
from Data_Pipeline.dataset import create_dataloaders
 # 创建 DataLoader
print("创建 DataLoader...")
data_dict = {'lob': lob_data,'trade': trade_data}
train_loader, val_loader = create_dataloaders(
    data_dict=data_dict,
    labels=labels_class,
    returns=labels_ret,
    config=all_config,
    device=device
)

print(f"训练集大小: {len(train_loader.dataset)}")
print(f"验证集大小: {len(val_loader.dataset)}")

创建 DataLoader...
训练集大小: 1174968
验证集大小: 293698


In [16]:
# ## 打印train_loader 的第一个
# print(next(iter(train_loader))[0].keys())

In [17]:
for inputs,labels,returns in train_loader:
    print(inputs['lob'].shape,inputs['trade'].shape)
    break

torch.Size([32, 1, 1200, 40]) torch.Size([32, 23, 1200])


### 模型创建

##### 1.多模态模型

In [18]:
# # 创建模型
# print("创建模型...")

# from Model import MultiModalTransformer
# # 根据数据情况调整配置
# lob_config = model_config.get('lob_encoder', {})
# trade_config = model_config.get('trade_encoder') 
# fusion_config = model_config.get('fusion', {})
# transformer_config = model_config.get('transformer', {})
# output_config = model_config.get('output_head', {})

# model = MultiModalTransformer(
#     lob_config=lob_config,
#     trade_config=trade_config,
#     fusion_config=fusion_config,
#     transformer_config=transformer_config,
#     output_config=output_config,
#     use_revin=True
# )


In [19]:
from Model.encoders.deeplob_encoder import Deeplob_encoder
import torch
import torch.nn as nn
from torchinfo import summary
# ===================== 1. 定义 膨胀因果卷积1D 基础模块 =====================
class DilatedCausalConv1d(nn.Module):
    """
    膨胀因果卷积层（核心：保证因果性，不使用未来信息）
    :param in_channels: 输入通道数
    :param out_channels: 输出通道数
    :param kernel_size: 卷积核大小
    :param dilation: 膨胀率
    """
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, dilation: int = 1):
        super().__init__()
        # 因果卷积关键：padding = (kernel_size - 1) * dilation
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=self.padding
        )
        # 激活函数+归一化（和原编码器保持一致）
        self.act = nn.LeakyReLU(negative_slope=0.01)
        self.bn = nn.BatchNorm1d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        # 裁剪右侧padding，保证因果性（只保留历史信息）
        x = x[:, :, :-self.padding] if self.padding > 0 else x
        x = self.act(x)
        x = self.bn(x)
        return x
# ===================== 2. 完整预测模型：编码器 + 膨胀因果卷积头 =====================
class LOB_DilatedConv_Predictor(nn.Module):
    def __init__(self, num_classes: int = 1):
        """
        :param num_classes: 预测输出维度（1=回归，3=三分类，自定义）
        """
        super().__init__()
        # 1. 特征编码器（你的原有模型）
        self.encoder = Deeplob_encoder()
        self.encoder_dim = self.encoder.output_dim  # 192

        self.fusion = nn.Linear(self.encoder_dim, 64)


        # 2. 多层膨胀因果卷积（dilation指数增长，扩大感受野）
        self.dilated_conv_layers = nn.Sequential(
            DilatedCausalConv1d(in_channels=64, out_channels=32, kernel_size=3, dilation=1),
            DilatedCausalConv1d(in_channels=32, out_channels=32, kernel_size=3, dilation=2),
            DilatedCausalConv1d(in_channels=32, out_channels=16, kernel_size=3, dilation=4),
            # DilatedCausalConv1d(in_channels=32, out_channels=16, kernel_size=3, dilation=8),
        )

        # 3. 预测头（自适应池化 + 全连接层）
        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)  # 压缩时序维度为1
        self.fc = nn.Linear(16, num_classes)

    def forward(self, inputs):
        """
        输入x形状：[Batch, Channel=1, Height, Width] (LOB标准输入)
        输出：[Batch, num_classes] (预测结果)
        """
        x = inputs['lob']
        # ========== 步骤1：编码器提取特征 ==========
        feat = self.encoder(x)  # 输出: [B, T, 192]

        feat = self.fusion(feat)

        # ========== 步骤2：维度转换（适配Conv1d） ==========
        # Conv1d要求输入: [Batch, Channels, Length] → 转置 [B, 192, T]
        feat = feat.permute(0, 2, 1)

        # ========== 步骤3：膨胀因果卷积提取时序特征 ==========
        feat = self.dilated_conv_layers(feat)  # [B, 16, T]

        # ========== 步骤4：池化 + 预测 ==========
        feat = self.adaptive_pool(feat).squeeze(-1)  # [B, 16]
        out = self.fc(feat)  # [B, num_classes]

        return out

model = LOB_DilatedConv_Predictor(num_classes = 3)
# summary(model,(1,1,100,40))

In [20]:

import torch
import torch.nn as nn
from torchinfo import summary
# 3. 构造输入张量（维度需匹配配置）
for inputs,labels,returns in train_loader:
    lob_input = inputs['lob']
    trade_input = inputs['trade']
    break
# 4. 封装模型：将字典输入转为位置参数（适配torchinfo）
class WrappedMultiModalModel(nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.original_model = original_model
    
    def forward(self, lob, trade=None):
        # 构造模型需要的字典输入
        if trade is not None:
            inputs = {"lob": lob,"trade":trade}
        else:
            inputs = {"lob": lob}
        return self.original_model(inputs)

# inputs = {'lob': lob_input}
# with torch.no_grad():
#     model(inputs)  # 这一步后，self.fusion 不再是 None
wrapped_model = WrappedMultiModalModel(model)

# 5. 调用summary（核心：传入输入张量列表，顺序匹配封装模型的forward参数）
summary(
    wrapped_model,
    input_data=[lob_input,trade_input],  # 先lob，后trade
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    depth=5,  # 显示模型深度（层数）
    device="cuda"  # 若用GPU，改为"cuda"（需确保张量在GPU上）
)


Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
WrappedMultiModalModel                   [32, 1, 1200, 40]    [32, 3]              --                   True
├─LOB_DilatedConv_Predictor: 1-1         [32, 1, 1200, 40]    [32, 3]              --                   True
│    └─Deeplob_encoder: 2-1              [32, 1, 1200, 40]    [32, 1182, 192]      --                   True
│    │    └─Sequential: 3-1              [32, 1, 1200, 40]    [32, 32, 1194, 20]   --                   True
│    │    │    └─Conv2d: 4-1             [32, 1, 1200, 40]    [32, 32, 1200, 20]   96                   True
│    │    │    └─LeakyReLU: 4-2          [32, 32, 1200, 20]   [32, 32, 1200, 20]   --                   --
│    │    │    └─BatchNorm2d: 4-3        [32, 32, 1200, 20]   [32, 32, 1200, 20]   64                   True
│    │    │    └─Conv2d: 4-4             [32, 32, 1200, 20]   [32, 32, 1197, 20]   4,128                True
│    │    │    └

In [21]:

# # 3. 构造输入张量（维度需匹配配置）
# batch_size = 2
# time_steps = all_config.get('data', {}).get('history_T', 3000)  # LOB/Trade的时间步必须一致
# lob_dim = model_config.get('lob_encoder', {}).get('in_channels', 4)
# # trade_dim = model_config.get('trade_encoder', {}).get('in_features', 12)
# trade_dim = 12
# lob_input = torch.randn(batch_size, lob_dim, time_steps, 10,device=device)  # (B,C,T,L) = (2,4,10,20)
# trade_input = torch.randn(batch_size, trade_dim, time_steps,device=device)    # (B,F,T) = (2,26,10)

# 4. 封装模型：将字典输入转为位置参数（适配torchinfo）
class WrappedMultiModalModel(nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.original_model = original_model
    
    def forward(self, lob, trade):
        # 构造模型需要的字典输入
        inputs = {"lob": lob,"trade":trade}
        return self.original_model(inputs)

# inputs = {'lob': lob_input}
# with torch.no_grad():
#     model(inputs)  # 这一步后，self.fusion 不再是 None
wrapped_model = WrappedMultiModalModel(model)
model_summary_str = str(summary(
    wrapped_model,
    input_data=[lob_input, trade_input],
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    depth=4,
    device="cuda"
))
import matplotlib.pyplot as plt
# 2. 用 Matplotlib 绘制文本图
plt.figure(figsize=(20, 25))  # 根据模型长度调整
plt.text(0.01, 0.99, model_summary_str, fontsize=10, verticalalignment='top', family='monospace')
plt.axis('off')
plt.tight_layout()

# 3. 保存图片
model_summary_save_path = f"/root/lio/Trade_LOB_MultiModal/checkpoints/{model_version}"
model_summary_save_path = Path(model_summary_save_path)
plt.savefig(os.path.join(model_summary_save_path, f'{model_version}_model_summary.png'), 
            dpi=150, bbox_inches='tight')
plt.close()
# 5. 调用summary（核心：传入输入张量列表，顺序匹配封装模型的forward参数）,保存为图片
# summary_img = summary(
#     wrapped_model,
#     input_data=[lob_input, trade_input],  # 先lob，后trade
#     col_names=["input_size", "output_size", "num_params", "trainable"],
#     col_width=20,
#     depth=5,  # 显示模型深度（层数）
#     device="cuda"  # 若用GPU，改为"cuda"（需确保张量在GPU上）
# )
# summary_img.savefig(os.path.join(self.output_dir, f'{variant_name}_model_summary.png'))

In [22]:
# 打印模型信息
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型参数量: {total_params:,} (可训练: {trainable_params:,})")

模型参数量: 101,059 (可训练: 101,059)


### 训练

In [23]:
import os
import torch

# # ===================== 日志关闭核心代码 (所有PyTorch版本通用) =====================
# # 关闭 torch.compile 的 AUTOTUNE 满屏刷屏日志 (重中之重)
# os.environ['TORCHINDUCTOR_PRINT_CONFIG'] = '0'
# os.environ['TORCHINDUCTOR_VERBOSE'] = '0'
# os.environ['TORCHINDUCTOR_AUTOTUNE_LOG'] = '0'
# # 关闭PyTorch编译器的所有警告/错误日志输出
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# os.environ['TORCH_LOGS'] = '0'

#'default'  , 'max-autotune'  , 'reduce-overhead'
print("Compiling model...")
model = torch.compile(model, mode='reduce-overhead') 

Compiling model...


In [24]:
from Train.trainer import Trainer
from Train.train_utils import (
    setup_optimizer,
    setup_scheduler,
    setup_loss_functions,
    # set_seed
)
# 设置优化器和调度器
optimizer = setup_optimizer(model, train_config.get('optimizer', {}))
scheduler = setup_scheduler(optimizer, train_config.get('scheduler', {}))

# 设置损失函数
loss_fn = setup_loss_functions(train_config.get('loss', {}), device=device)


In [25]:

# 创建训练器
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    config=train_config,
    device=device,
    variant_name=model_version,
    seed=42
)

# 开始训练和验证
print("=" * 50)
history = trainer.fit()

## 保存为pickle
import pickle
history_save_path = f"/root/lio/Trade_LOB_MultiModal/checkpoints/{model_version}"
history_save_path = Path(history_save_path)
with open(os.path.join(history_save_path, f'{model_version}_history.pkl'), 'wb') as f:
    pickle.dump(history, f)

print('history保存成功')

print("=" * 50)
print("训练完成!")

# print(f"最佳验证 F1 (Up/Down): {max(history['val_f1_updown']):.4f}")

开始训练，共 3 个 epoch
设备: cuda
AMP: True
梯度累积步数: 1
--------------------------------------------------


Training:   0%|          | 0/36717 [00:00<?, ?it/s]

Epoch 1/3 (455.9s)
  Train Loss: 0.3950, Train Acc: 0.5952
  Val Loss: 0.3479, Val Acc: 0.6728
  Val F1 (Macro): 0.3147
  Val Precision (Up/Down): 0.3865, Recall (Up/Down): 0.0380, F1 (Up/Down): 0.0692
  [NEW BEST] loss: 0.3479



KeyboardInterrupt: 

## 测试结果

In [ ]:
from pathlib import Path
save_dir = Path('/root/autodl-tmp/train_data/')
save_dir.mkdir(parents=True,exist_ok=True)


In [ ]:
## 加载模型
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_version = 'multi_modal_model_1'
model_path = f'/root/lio/Trade_LOB_MultiModal/checkpoints/{model_version}/seed_42/best_model.pt'
model = torch.load(model_path,map_location=device)
model.eval()

## 加载数据
lob_data_test = np.load('/root/autodl-tmp/test_data/lob_data.npy')
trade_data_test = np.load('/root/autodl-tmp/test_data/trade_data.npy')
labels_ret_test = np.load('/root/autodl-tmp/test_data/trade_labels_ret.npy')
time_buckt = np.load('/root/autodl-tmp/test_data/time_bucket.npy')

print(f"trade_data.shape: {trade_data_test.shape}")
print(f"lob_data.shape: {lob_data_test.shape}")
print(f"labels_ret.shape: {labels_ret_test.shape}")

alpha= 0.001
labels_class_test = np.ones_like(labels_ret_test, dtype=np.int8)
# 3. 向量化赋值：涨→2，跌→0
# 涨：labels_ret > alpha
labels_class_test[labels_ret_test > alpha] = 2
# 跌：labels_ret < -alpha
labels_class_test[labels_ret_test < -alpha] = 0


NameError: name 'torch' is not defined

In [ ]:
## 创建数据集
from Data_Pipeline.dataset import create_dataloaders_for_test
test_data_dict = {'lob': lob_data_test,
                    'trade': trade_data_test}

## 加载配置
all_config = '/root/lio/Trade_LOB_MultiModal/Configs/experiment_config.yaml'
with open(all_config, 'r') as f:
    all_config = yaml.safe_load(f)
test_config ={
    'history_T': all_config.get('data', {}).get('history_T', 3000),
    'batch_size': all_config.get('data', {}).get('batch_size', 1),
    'num_workers': 4,
    'pin_memory': True,
    'stride': 100,  ## 回测间隔
}

test_loader = create_dataloaders_for_test(test_data_dict,
                                            labels=labels_class_test,
                                            returns=labels_ret_test,
                                            config=test_config,
                                            device=device)
## 测试
pred_classes = []
pred_probas = []
actual_returns = []
tick_indices = []
prices = []
use_amp = True

from torch.amp import autocast
from tqdm import tqdm
with torch.no_grad():
    for inputs, _, returns in tqdm(test_loader, desc="Testing", leave=False):
        if not isinstance(inputs, dict):
            raise TypeError("输入格式异常，期望 dict[str, Tensor]。")
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

        with autocast(device_type="cuda", enabled=use_amp):
            logits = model(inputs)
            probas = torch.softmax(logits, dim=1)

        # preds = torch.argmax(probas, dim=1).cpu().numpy()
        probas_np = probas.cpu().numpy()
        returns_np = returns.numpy()
        pred_probas.append(probas_np)
        actual_returns.append(returns_np)
    pred_probas_arr = np.concatenate(pred_probas, axis=0)
    actual_return_arr = np.concatenate(actual_returns, axis=0)

signal_df = pl.DataFrame(
    {
        'pred_down': pred_probas_arr[:, 0],
        'pred_stationary': pred_probas_arr[:, 1],
        'pred_up': pred_probas_arr[:, 2],
        'actual_return': actual_return_arr,
    }
)
if signal_df.shape[0] != len(labels_ret_test):
    raise ValueError("信号数量与标签数量不一致")
test_signal_path = '/root/lio/Trade_LOB_MultiModal/checkpoints/multi_modal_model_1/seed_42/signal.csv'
signal_df.write_csv(test_signal_path)
